In [3]:
import os 
import pandas as pd
import numpy as np
import re as re
import networkx as nx
from rapidfuzz import fuzz
from collections import defaultdict
import itertools
 

In [22]:
names = pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/names_consolidated.csv')

In [23]:
names['standardized_org_name'] = names['standardized_org_name'].str.strip().str.lower()

pattern = r'^(a|an)\s+\w+\s+(corporation|corp|llc|limited|company|incorporated|inc|partnership|lp|llp)\.?$'

mask = names['standardized_org_name'].str.match(pattern, case=False, na=False)
print(names[mask]['standardized_org_name'].unique())

names['standardized_org_name'] = names['standardized_org_name'].str.strip().apply(
    lambda x: 'Non-Specific Entity' if re.match(pattern, str(x), re.IGNORECASE) else x
)

<StringArray>
[      'a limited partnership',     'an illinois partnership',
    'a california partnership', 'a massachusetts partnership',
         'a texas partnership',  'a professional partnership',
        'an irish partnership',         'an ohio partnership',
      'a canadian partnership',      'a delaware partnership',
      'a colorado partnership',          'a utah partnership',
    'a washington partnership',      'an arizona partnership',
         'a calif partnership',  'a pennsylvania partnership',
            'a washington llp',       'a general partnership',
       'an oregon partnership',      'an israeli partnership',
       'an israel partnership',           'a law partnership',
       'a florida partnership']
Length: 23, dtype: str


In [25]:
from rapidfuzz import fuzz

def is_place_entity(name):
    if not isinstance(name, str):
        return False
    words = name.strip().lower().split()
    if words[0] not in ('a', 'an'):
        return False
    # Check if any word in the name fuzzy matches a known place/state name
    # and ends with a legal suffix
    legal_suffixes = {'corp', 'corp.', 'corporation', 'llc', 'inc', 'inc.', 
                      'limited', 'company', 'incorporated', 'partnership', 'lp', 'llp'}
    if not any(w.rstrip('.') in legal_suffixes for w in words):
        return False
    # Check middle words for fuzzy match against known places
    states = ['california', 'delaware', 'idaho', 'nevada', 'texas', 
              'florida', 'new york', 'illinois', 'ohio', 'washington']
    for word in words[1:]:
        for state in states:
            if fuzz.ratio(word.rstrip(','), state) >= 80:
                return True
    return False

# Check first
mask = names['standardized_org_name'].apply(is_place_entity)
print(names[mask]['standardized_org_name'].unique())

# Apply
names.loc[mask, 'standardized_org_name'] = 'Non-Specific Entity'

<StringArray>
[]
Length: 0, dtype: str


In [26]:
clusters = pd.read_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/clusters_for_review.csv')
 
print(f"Rows in dataset: {len(names):,}")
print(f"Unique names before consolidation: {names['standardized_org_name'].nunique():,}")
 
 
# ============================================================
# STEP 2: BUILD CONSOLIDATION MAP FROM YOUR CANON NAMES
# ============================================================
# For every cluster where you defined a canonical name,
# map all variants in that cluster -> your canonical name
 
consolidation_map = {}
 
has_canon = clusters[clusters['canon name'].notna() & (clusters['canon name'].str.strip() != '')]
 
for cluster_id, group in has_canon.groupby('cluster_id'):
    canon = group['canon name'].iloc[0].strip()
    # Get ALL variants in this cluster (not just the ones with canon filled in)
    all_variants = clusters[clusters['cluster_id'] == cluster_id]['name'].tolist()
    for variant in all_variants:
        variant = variant.strip()
        if variant != canon:
            consolidation_map[variant] = canon
 
print(f"Clusters with canonical names defined: {has_canon['cluster_id'].nunique():,}")
print(f"Total variant -> canonical mappings: {len(consolidation_map):,}")
 
 
# ============================================================
# STEP 3: APPLY CONSOLIDATION
# ============================================================
 
names['standardized_org_name'] = names['standardized_org_name'].str.strip()
names['standardized_org_name'] = names['standardized_org_name'].replace(consolidation_map)
 
print(f"Unique names after consolidation:  {names['standardized_org_name'].nunique():,}")
 
 
# ============================================================
# STEP 4: SPOT CHECK — show what changed
# ============================================================
 
print("\nSample of consolidations applied:")
for variant, canon in list(consolidation_map.items())[:20]:
    print(f"  \"{variant}\"")
    print(f"    -> \"{canon}\"")
 
 
# ============================================================
# STEP 5: SAVE OUTPUT
# ============================================================
 
output_path = '/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/names_consolidated.csv'
names.to_csv(output_path, index=False)
print(f"\nSaved to: {output_path}")
 
 
# ============================================================
# STEP 6: SHOW UNCOVERED CLUSTERS
# ============================================================
# Clusters in the CSV that still don't have a canonical name
 
no_canon = clusters[clusters['canon name'].isna() | (clusters['canon name'].str.strip() == '')]
uncovered = no_canon['cluster_id'].nunique()
print(f"\nClusters still without a canonical name: {uncovered:,}")
print("Open clusters_for_review.csv and fill in the 'canon name' column to cover more.")
 

Rows in dataset: 559,732
Unique names before consolidation: 89,403
Clusters with canonical names defined: 121
Total variant -> canonical mappings: 507
Unique names after consolidation:  88,970

Sample of consolidations applied:
  "matsushita electric industrial co. ltd.,"
    -> "matsushita electric"
  "matsushita electric industrial company., ltd."
    -> "matsushita electric"
  "matsushita electrical industrial co, ltd."
    -> "matsushita electric"
  "matsushita electrical industrial co., ltd.,"
    -> "matsushita electric"
  "matsushita electrical industrial company, ltd."
    -> "matsushita electric"
  "matsushita electrical industrial, co."
    -> "matsushita electric"
  "matsushita electronics industrial co, ltd."
    -> "matsushita electric"
  "matsushita electronics industrial co., ltd."
    -> "matsushita electric"
  "matsushita industrial electric co., ltd."
    -> "matsushita electric"
  "under secretary of commerce for intellectual property an director of the united states

In [ ]:
name_list = names['standardized_org_name'].dropna().unique().tolist()
print(f"Total unique names to cluster: {len(name_list)}")

 
def get_block_key(name):
    """Return the first meaningful word of a name as the block key."""
    words = str(name).strip().split()
    return words[0].lower() if words else ''
 

blocks = defaultdict(list)
for name in name_list:
    key = get_block_key(name)
    blocks[key].append(name)
 
# Report on blocking
block_sizes = {k: len(v) for k, v in blocks.items() if len(v) > 1}
print(f"Blocks with 2+ names: {len(block_sizes)}")
print(f"Largest block: '{max(block_sizes, key=block_sizes.get)}' with {max(block_sizes.values())} names")
 

 
THRESHOLD = 90  # Adjust this: higher = stricter, lower = more matches
 
G = nx.Graph()
G.add_nodes_from(name_list)
 
total_pairs_checked = 0
edges_added = 0
 
for block_key, block_names in blocks.items():
    if len(block_names) < 2:
        continue  # skip singletons
    
    for a, b in itertools.combinations(block_names, 2):
        total_pairs_checked += 1
        score = fuzz.token_sort_ratio(a, b)
        if score >= THRESHOLD:
            G.add_edge(a, b, weight=score)
            edges_added += 1
 
print(f"\nPairs checked: {total_pairs_checked:,}")
print(f"Similar pairs found: {edges_added:,}")
 

 
clusters = list(nx.connected_components(G))
multi_clusters = [c for c in clusters if len(c) > 1]
 
print(f"\nTotal clusters found: {len(clusters):,}")
print(f"Clusters with 2+ names (need review): {len(multi_clusters):,}")
 

 
cluster_rows = []
for i, cluster in enumerate(sorted(multi_clusters, key=len, reverse=True)):
    for name in sorted(cluster):
        cluster_rows.append({
            'cluster_id': i,
            'cluster_size': len(cluster),
            'name': name
        })
 
cluster_df = pd.DataFrame(cluster_rows)
print("\nSample of clusters found:")
print(cluster_df.head(30).to_string(index=False))

 
print("\n" + "="*60)
print("CLUSTER REVIEW (largest clusters first)")
print("="*60)
 
for i, cluster in enumerate(sorted(multi_clusters, key=len, reverse=True)):
    print(f"\nCluster {i} ({len(cluster)} names):")
    for name in sorted(cluster):
        print(f"    {name}")
    
    if i >= 49:  # Print first 50 clusters then stop
        remaining = len(multi_clusters) - 50
        print(f"\n... and {remaining} more clusters. See cluster_df for full list.")
        break

 
output_path = '/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/clusters_for_review.csv'
cluster_df.to_csv(output_path, index=False)
print(f"\nFull cluster list saved to: {output_path}")


 
if CONSOLIDATION_MAP:
    names['standardized_org_name'] = names['standardized_org_name'].replace(CONSOLIDATION_MAP)
    print(f"Applied {len(CONSOLIDATION_MAP)} consolidation mappings to 'standardized_org_name' column.")
else:
    print("No consolidation mappings applied yet — add entries to CONSOLIDATION_MAP after reviewing clusters.")
 

Total unique names to cluster: 88970
Blocks with 2+ names: 10435
Largest block: 'the' with 1287 names

Pairs checked: 2,264,995
Similar pairs found: 2,961

Total clusters found: 86,125
Clusters with 2+ names (need review): 2,603

Sample of clusters found:
 cluster_id  cluster_size                                                                          name
          0             7                                                           the boc group, inc.
          0             7                                                           the box group, inc.
          0             7                                                         the cbord group, inc.
          0             7                                                         the decor group, inc.
          0             7                                                        the encore group, inc.
          0             7                                                       the fidelco group, inc.
          0     

In [30]:
# Get row counts for every unique name
row_counts = names['standardized_org_name'].value_counts().to_dict()

# Build cluster output with row counts
print("="*60)
print("CLUSTER REVIEW (by total rows affected)")
print("="*60)

cluster_rows = []
for i, cluster in enumerate(sorted(multi_clusters, key=lambda c: sum(row_counts.get(n, 0) for n in c), reverse=True)):
    total_rows = sum(row_counts.get(n, 0) for n in cluster)
    for name in sorted(cluster, key=lambda n: row_counts.get(n, 0), reverse=True):
        cluster_rows.append({
            'cluster_id': i,
            'cluster_size': len(cluster),
            'cluster_total_rows': total_rows,
            'name': name,
            'name_row_count': row_counts.get(name, 0)
        })

cluster_df = pd.DataFrame(cluster_rows)
cluster_df.to_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/clusters_for_review2.csv', index=False)
print(f"Saved {len(cluster_df)} rows to clusters_for_review.csv")

for i, cluster in enumerate(sorted(multi_clusters, key=lambda c: sum(row_counts.get(n, 0) for n in c), reverse=True)):
    total_rows = sum(row_counts.get(n, 0) for n in cluster)
    print(f"\nCluster {i} ({len(cluster)} variants, {total_rows:,} total rows):")
    for name in sorted(cluster, key=lambda n: row_counts.get(n, 0), reverse=True):
        print(f"    {row_counts.get(name, 0):>6,} rows  |  {name}")

CLUSTER REVIEW (by total rows affected)
Saved 5448 rows to clusters_for_review.csv

Cluster 0 (2 variants, 256 total rows):
       255 rows  |  laughlin products, inc.
         1 rows  |  laughlin prod, inc.

Cluster 1 (2 variants, 232 total rows):
       231 rows  |  pantech co. ltd.
         1 rows  |  pantech co.,ltd.

Cluster 2 (2 variants, 231 total rows):
       219 rows  |  uniloc luxembourg s.a.
        12 rows  |  uniloc luxembourg, s.a.

Cluster 3 (2 variants, 215 total rows):
       214 rows  |  thermolife international, llc
         1 rows  |  thermolife international

Cluster 4 (2 variants, 199 total rows):
       198 rows  |  nissan motor co.
         1 rows  |  nissan motor, corp.

Cluster 5 (2 variants, 186 total rows):
       107 rows  |  intellectual ventures ii, llc
        79 rows  |  intellectual ventures i, llc

Cluster 6 (2 variants, 183 total rows):
       175 rows  |  garmin ltd.
         8 rows  |  garmin ltd.,

Cluster 7 (2 variants, 176 total rows):
       1

In [ ]:

all_counts = names['standardized_org_name'].value_counts().reset_index()
all_counts.columns = ['name', 'row_count']


clustered_names = set(name for cluster in multi_clusters for name in cluster)


all_counts['has_variants'] = all_counts['name'].isin(clustered_names)

print(all_counts.head(50).to_string(index=False))


all_counts.to_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/all_names_with_counts.csv', index=False)

                                    name  row_count  has_variants
                     non-specific entity      45414         False
                     samsung electronics       2237         False
                            actavis inc.       1795         False
                              sony corp.       1701         False
                          lg electronics       1397         False
          teva pharmaceutical industries       1329         False
                             pfizer inc.       1198         False
                              mylan inc.       1188         False
                              apple inc.       1104         False
                               at&t inc.       1101         False
                  verizon communications       1098         False
                     Non-Specific Entity       1094         False
                        arrivalstar s.a.       1056         False
                             novartis ag        951         False
          

In [ ]:


REVIEW_THRESHOLD = 0.5  

auto_consolidate = {}  
needs_review = {}       
for cluster_id, group in clusters.groupby('cluster_id'):
    group = group.sort_values('name_row_count', ascending=False)
    top_name = group.iloc[0]['name']
    top_count = group.iloc[0]['name_row_count']
    
    if len(group) == 1:
        continue  
    
    second_count = group.iloc[1]['name_row_count']
    ratio = second_count / top_count if top_count > 0 else 1.0
    
    if ratio <= REVIEW_THRESHOLD:
      
        auto_consolidate[cluster_id] = top_name
    else:
    
        needs_review[cluster_id] = group['name'].tolist()

print(f"Clusters safe to auto-consolidate: {len(auto_consolidate):,}")
print(f"Clusters flagged for manual review: {len(needs_review):,}")




MANUAL_OVERRIDES = {
   
    
    7:   None,  
    13:  "watson laboratories, inc. - florida",
    14:  "icon health & fitness, inc.",
    23:  "home depot u.s.a., inc.",
    31:  None,  e
    36:  "skechers usa, inc.",
    43:  None,  
    47:  "takeda pharmaceuticals usa, inc.",
    52:  "charles e. hill & associates, inc.",
    64:  "elonex ip holdings, ltd.",
    66:  None,  
    68:  "Non-Specific Entity", 
   
}




consolidation_map = {}

for cluster_id, canonical in auto_consolidate.items():
    group = clusters[clusters['cluster_id'] == cluster_id]
    for _, row in group.iterrows():
        if row['name'] != canonical:
            consolidation_map[row['name']] = canonical

# Manual override clusters
for cluster_id, canonical in MANUAL_OVERRIDES.items():
    if canonical is None:
        continue  # skip this cluster
    group = clusters[clusters['cluster_id'] == cluster_id]
    for _, row in group.iterrows():
        if row['name'] != canonical:
            consolidation_map[row['name']] = canonical

print(f"\nTotal name variants to be consolidated: {len(consolidation_map):,}")




before_unique = names['standardized_org_name'].nunique()
names['standardized_org_name'] = names['standardized_org_name'].replace(consolidation_map)
after_unique = names['standardized_org_name'].nunique()

print(f"\nUnique names before consolidation: {before_unique:,}")
print(f"Unique names after consolidation:  {after_unique:,}")
print(f"Names consolidated: {before_unique - after_unique:,}")




output_path = '/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/names_consolidated.csv'
names.to_csv(output_path, index=False)
print(f"\nSaved consolidated data to: {output_path}")



unresolved = {k: v for k, v in needs_review.items() if k not in MANUAL_OVERRIDES}
print(f"\n{'='*60}")
print(f"STILL NEEDS MANUAL REVIEW: {len(unresolved)} clusters")
print(f"{'='*60}")
print("Add these to MANUAL_OVERRIDES with your canonical choice:\n")

for cluster_id, variants in list(unresolved.items())[:50]:  # show first 50
    cluster_rows = clusters[clusters['cluster_id'] == cluster_id].sort_values('name_row_count', ascending=False)
    total = cluster_rows['cluster_total_rows'].iloc[0]
    print(f"Cluster {cluster_id} ({total} total rows):")
    for _, row in cluster_rows.iterrows():
        print(f"    {row['name_row_count']:>5} rows  |  {row['name']}")
    print()

if len(unresolved) > 50:
    print(f"... and {len(unresolved) - 50} more. Work through them in clusters_for_review.csv")

Clusters safe to auto-consolidate: 1,601
Clusters flagged for manual review: 1,129

Total name variants to be consolidated: 1,903

Unique names before consolidation: 92,297
Unique names after consolidation:  92,283
Names consolidated: 14

Saved consolidated data to: /Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/names_consolidated.csv

STILL NEEDS MANUAL REVIEW: 1117 clusters
Add these to MANUAL_OVERRIDES with your canonical choice:

Cluster 72 (63 total rows):
       33 rows  |  utstarcom, inc.
       30 rows  |  utstarcom, inc.,

Cluster 74 (61 total rows):
       38 rows  |  a united kingdom, corp.
       23 rows  |  a united kingdom, co.

Cluster 81 (59 total rows):
       37 rows  |  acting under secretary of commerce for intellectual property and acting director of the united states patent and trademark office
       21 rows  |  acting under secretary of commerce for intellectual property and director of the united states patent and trademark office
        1

In [ ]:
names.to_csv('/Users/matt/Desktop/Patent Litigation Analysis/data/processed/names/names_cleaned_v7.csv', index=False)